In [17]:
import pandas as pd
import json
import cv2
import numpy as np
from pathlib import Path
import os

In [37]:
video_path = r"/home/share/schaer2/idtracking_keypoint/input/20200505155636_20200505174320_0_converted_small.mp4"
# bbox = r"/home/share/schaer2/idtracking_keypoint/output/20200505155636_20200505174320_0_converted_small_bbox.json"
# skpoint = r"/home/share/schaer2/idtracking_keypoint/output/results_20200505155636_20200505174320_0_converted_small.json"
timeline_behavior = r"/home/share/schaer2/idtracking_keypoint/output/8137_data_Timelight_segment.csv"
start = 210
end = 360

In [38]:
timeline_behavior = pd.read_csv(timeline_behavior)
timeline_behavior

,behavior_category,behavior,modifier,event_type,start,end
0,Gestures,Rep-mov,Bras Gauche,state,328.12,333.88
1,Gestures,Rep-mov,Bras Droit,state,246.52,247.92
2,Gestures,Rep-mov,Bras Droit,state,320.80,330.00


In [ ]:

timeline_behavior.start = timeline_behavior.start - start
timeline_behavior.end = timeline_behavior.end - start

In [31]:
timeline_behavior

,behavior_category,behavior,modifier,event_type,start,end
0,Gestures,Rep-mov,Bras Gauche,state,118.12,123.88
1,Gestures,Rep-mov,Bras Droit,state,36.52,37.92
2,Gestures,Rep-mov,Bras Droit,state,110.80,120.00


In [35]:
# Clean implementation for 3 individuals per frame


# OpenPose COCO connections for skeleton
COCO_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)
]

# Colors for 3 individuals (BGR format)
COLORS = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]  # Blue, Green, Red

COLORS = [
    (255, 0, 0),    # Red
    (0, 255, 0),    # Green
    (0, 0, 255),    # Blue
    (255, 255, 0),  # Cyan
    (255, 0, 255),  # Magenta
    (0, 255, 255),  # Yellow
    (128, 0, 0),    # Maroon
    (0, 128, 0),    # Dark Green
    (0, 0, 128),    # Navy
    (128, 128, 0),  # Olive
    (128, 0, 128),  # Purple
    (0, 128, 128),  # Teal
    (192, 192, 192)] # Silver]

# Behavior colors
BEHAVIOR_COLORS = {mod: COLORS[2:][i % len(COLORS[2:])] for i, mod in enumerate(timeline_behavior['modifier'].unique())}


def draw_skeleton(image, keypoints, connections, color, threshold=0.3):
    """Draw skeleton keypoints and connections"""
    img = image.copy()
    h, w = img.shape[:2]
    
    if keypoints is None or len(keypoints) == 0:
        return img
    
    # Draw connections
    for connection in connections:
        idx1, idx2 = connection
        if (idx1 < len(keypoints) and idx2 < len(keypoints) and 
            keypoints[idx1][2] > threshold and keypoints[idx2][2] > threshold):
            x1, y1 = int(keypoints[idx1][0]), int(keypoints[idx1][1])
            x2, y2 = int(keypoints[idx2][0]), int(keypoints[idx2][1])
            cv2.line(img, (x1, y1), (x2, y2), color, 2)
    
    # Draw keypoints
    for x, y, conf in keypoints:
        if conf > threshold:
            cv2.circle(img, (int(x), int(y)), 4, color, -1)
    
    return img

def draw_bbox(image, bbox, color, is_normalized=True):
    """Draw bounding box"""
    img = image.copy()
    h, w = img.shape[:2]
    
    if bbox is None:
        return img

    x1 = bbox[0]
    y1 = bbox[1]
    x2 = bbox[2]
    y2 = bbox[3]
    
    if is_normalized:
        # Convert normalized coordinates to pixel values
        x1 = int(x1 * w)
        y1 = int(y1 * h)
        x2 = int(x2 * w)
        y2 = int(y2 * h)
        
        # Ensure coordinates are within image bounds
        x1 = max(0, min(x1, w - 1))
        y1 = max(0, min(y1, h - 1))
        x2 = max(0, min(x2, w - 1))
        y2 = max(0, min(y2, h - 1))

    
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    return img

def extract_frame_data(bbox_data, keypoint_data, frame_idx):
    """Extract bbox and keypoints for all individuals in a frame"""
    # Extract bboxes
    bboxes = []
    if str(frame_idx) in bbox_data:
        frame_bboxes = bbox_data[str(frame_idx)]
        for person_id in ['0', '1', '2']:  # 3 individuals
            if person_id in frame_bboxes:
                bboxes.append(frame_bboxes[person_id])
            else:
                bboxes.append(None)
    
    # Extract keypoints
    keypoints_list = []
    if 'instance_info' in keypoint_data:
        # Find frame data
        frame_data = None
        for data in keypoint_data['instance_info']:
            if data['frame_id'] == frame_idx:
                frame_data = data
                break
        
        if frame_data and 'instances' in frame_data:
            instances = frame_data['instances']
            for i in range(3):  # 3 individuals
                if i < len(instances):
                    person = instances[i]
                    if 'keypoints' in person and 'keypoint_scores' in person:
                        kp = np.array(person['keypoints'])  # (17, 2)
                        scores = np.array(person['keypoint_scores'])  # (17,)
                        # Combine to (17, 3) format
                        combined = np.zeros((17, 3))
                        combined[:, :2] = kp
                        combined[:, 2] = scores
                        keypoints_list.append(combined)
                    else:
                        keypoints_list.append(None)
                else:
                    keypoints_list.append(None)
    
    return bboxes, keypoints_list
def draw_timeline_with_legends(width, height, current_time, total_duration, behavior_data, timeline_height=60):
    """Draw timeline with behavior annotations, time cursor, and legends"""
    # Create extended timeline with space for legends
    legend_height = 80
    total_height = timeline_height + legend_height
    timeline = np.zeros((total_height, width, 3), dtype=np.uint8)
    
    # Draw timeline background (dark gray)
    cv2.rectangle(timeline, (0, 0), (width, timeline_height), (40, 40, 40), -1)
    
    # Draw behavior rectangles
    for _, row in behavior_data.iterrows():
        start_time = row['start']
        end_time = row['end']
        modifier = row['modifier']

        
        # Convert time to pixel positions
        start_x = int((start_time / total_duration) * width)
        end_x = int((end_time / total_duration) * width)
        
        # Get color for behavior type
        color = BEHAVIOR_COLORS.get(modifier, (128, 128, 128))  # Default gray
        
        # Draw behavior rectangle
        cv2.rectangle(timeline, (start_x, 10), (end_x, timeline_height-10), color, -1)
        
        # Add behavior label
        label = modifier.replace('Bras ', '')  # Shorten label
        text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)[0]
        text_x = start_x + 2
        text_y = timeline_height // 2 + text_size[1] // 2
        if text_x + text_size[0] < end_x:  # Only draw if fits
            cv2.putText(timeline, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Draw time cursor (white vertical line)
    cursor_x = int((current_time / total_duration) * width)
    cv2.line(timeline, (cursor_x, 0), (cursor_x, timeline_height), (255, 255, 255), 2)
    
    # Draw time markers
    for i in range(0, int(total_duration), max(1, int(total_duration // 10))):
        marker_x = int((i / total_duration) * width)
        cv2.line(timeline, (marker_x, timeline_height-5), (marker_x, timeline_height), (200, 200, 200), 1)
        # Add time label
        time_label = f"{i//60}:{i%60:02d}" if i >= 60 else f"{i}s"
        cv2.putText(timeline, time_label, (marker_x+2, timeline_height-15), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (200, 200, 200), 1)
    
    # Draw legends below timeline
    legend_y_start = timeline_height + 15
    
    # Individual colors legend (left side)
    # legend_x = 20
    # cv2.putText(timeline, "Individuals:", (legend_x, legend_y_start), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # individual_labels = ["Child", "Clinician", "Parent"]
    # for i, (color, label) in enumerate(zip(COLORS, individual_labels)):
    #     y_pos = legend_y_start + 20 + (i * 18)
    #     # Draw color rectangle
    #     cv2.rectangle(timeline, (legend_x, y_pos-8), (legend_x+15, y_pos+2), color, -1)
    #     # Draw label
    #     cv2.putText(timeline, label, (legend_x + 20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Behavior colors legend (right side)
    behavior_legend_x = width // 2 + 50
    cv2.putText(timeline, "Repetitive Behaviors:", (behavior_legend_x, legend_y_start), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    behavior_items = [
        ("Bras Gauche", "Left Arm", BEHAVIOR_COLORS.get('Bras Gauche', (128, 128, 128))),
        ("Bras Droit", "Right Arm", BEHAVIOR_COLORS.get('Bras Droit', (128, 128, 128))),
        ("Shoulder", "Shoulder", BEHAVIOR_COLORS.get('Shoulder', (128, 128, 128)))
    ]
    
    for i, (label, color) in enumerate(BEHAVIOR_COLORS.items()):
        y_pos = legend_y_start + 20 + (i * 18)
        # Draw color rectangle
        cv2.rectangle(timeline, (behavior_legend_x, y_pos-8), (behavior_legend_x+15, y_pos+2), color, -1)
        # Draw label
        cv2.putText(timeline, label, (behavior_legend_x + 20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Add current time display (top right)
    time_text = f"Time: {int(current_time//60)}:{int(current_time%60):02d}"
    time_size = cv2.getTextSize(time_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
    cv2.putText(timeline, time_text, (10, legend_y_start + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    return timeline

def create_2x2_video(video_path, output_path, behavior_data, start=210, end=360):
    """Create 2x2 grid video with all 3 individuals"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Video dimensions: {width}x{height}, FPS: {fps}")
    print(f'focus on time {start}s to {end}s')
    

    timeline_height = 140
    output_width = width
    output_height = height + timeline_height

    # Output video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (output_width, output_height))

    max_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total_duration = end-start

    for frame_idx in range(max_frames):
        # Current time in seconds
        current_time = frame_idx / fps
        ret, frame = cap.read()
        if not ret:
            break

        if current_time < start or current_time > end:
            continue
        
        
        # Create 4 quadrants
        top = frame.copy()  # Original
        
        timeline = draw_timeline_with_legends(output_width, timeline_height, current_time-start, total_duration, behavior_data)
        # Combine video grid with timeline (timeline in the middle)
        final_frame = np.vstack((top, timeline))
        
        
        out.write(final_frame)
        
        if frame_idx % 50 == 0:
            print(f"Processed {frame_idx}/{max_frames} frames")
    
    cap.release()
    out.release()
    print(f"✅ Video saved: {output_path}")

# Load data and create test video
print("Loading data...")
# with open(bbox, 'r') as f:
#     bbox_data = json.load(f)
# with open(skpoint, 'r') as f:
#     keypoint_data = json.load(f)

# Create output path
output_path = r"/home/share/schaer2/idtracking_keypoint/output/test_focus.mp4"

print("Creating 10-second test video with 3 individuals...")
create_2x2_video(video_path, output_path, behavior_data=timeline_behavior, start=start, end=end)

Loading data...
Creating 10-second test video with 3 individuals...
Video dimensions: 480x270, FPS: 25.0
focus on time 210s to 360s
Processed 5250/45000 frames
Processed 5300/45000 frames
Processed 5350/45000 frames
Processed 5400/45000 frames
Processed 5450/45000 frames
Processed 5500/45000 frames
Processed 5550/45000 frames
Processed 5600/45000 frames
Processed 5650/45000 frames
Processed 5700/45000 frames
Processed 5750/45000 frames
Processed 5800/45000 frames
Processed 5850/45000 frames
Processed 5900/45000 frames
Processed 5950/45000 frames
Processed 6000/45000 frames
Processed 6050/45000 frames
Processed 6100/45000 frames
Processed 6150/45000 frames
Processed 6200/45000 frames
Processed 6250/45000 frames
Processed 6300/45000 frames
Processed 6350/45000 frames
Processed 6400/45000 frames
Processed 6450/45000 frames
Processed 6500/45000 frames
Processed 6550/45000 frames
Processed 6600/45000 frames
Processed 6650/45000 frames
Processed 6700/45000 frames
Processed 6750/45000 frames


In [21]:
# Verify the output
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
    print(f"\n✅ Success! Video created with 3 individuals:")
    print(f"📁 File: {output_path}")
    print(f"📏 Size: {file_size / (1024*1024):.2f} MB")
    
    # Check video properties
    cap = cv2.VideoCapture(output_path)
    test_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    test_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    test_fps = cap.get(cv2.CAP_PROP_FPS)
    test_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    print(f"🎥 Properties: {test_width}x{test_height} @ {test_fps} fps")
    print(f"⏱️  Duration: {test_frames/test_fps:.1f} seconds ({test_frames} frames)")
    
    print(f"\n🎨 Color coding:")
    print(f"   🔵 Person 1: Blue")
    print(f"   🟢 Person 2: Green")
    print(f"   🔴 Person 3: Red")
    
    print(f"\n📺 Layout:")
    print(f"   Top-left: Original video")
    print(f"   Top-right: Video + bounding boxes (3 colors)")
    print(f"   Bottom-left: Keypoints only (3 colors)")
    print(f"   Bottom-right: Video + keypoints (3 colors)")
else:
    print("❌ Video creation failed!")


✅ Success! Video created with 3 individuals:
📁 File: /home/share/schaer2/idtracking_keypoint/output/test_focus.mp4
📏 Size: 16.93 MB
🎥 Properties: 480x410 @ 25.0 fps
⏱️  Duration: 150.0 seconds (3751 frames)

🎨 Color coding:
   🔵 Person 1: Blue
   🟢 Person 2: Green
   🔴 Person 3: Red

📺 Layout:
   Top-left: Original video
   Top-right: Video + bounding boxes (3 colors)
   Bottom-left: Keypoints only (3 colors)
   Bottom-right: Video + keypoints (3 colors)


In [22]:
# Uncomment and run this cell to create the full video (all frames)
# WARNING: This will take much longer and create a large file

# full_output_path = str(Path(skpoint).parent / (Path(skpoint).stem + "_2x2_grid_FULL.mp4"))
# print("Creating FULL video with all 3 individuals...")
# print("This may take several minutes...")
# 
# # Get total video duration first
# cap = cv2.VideoCapture(video_path)
# total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# fps = cap.get(cv2.CAP_PROP_FPS)
# duration = total_frames / fps
# cap.release()
# 
# print(f"Full video: {duration:.1f} seconds ({total_frames} frames)")
# create_2x2_video(video_path, bbox_data, keypoint_data, full_output_path, max_seconds=int(duration)+1)

print("🎬 Ready! Review the 10-second test video first.")
print("If it looks good, uncomment the code above to create the full video.")

🎬 Ready! Review the 10-second test video first.
If it looks good, uncomment the code above to create the full video.
